# Section 1 Question 1: HDB Resale Portal and Property Agent Business Impact

**Question:** The HDB Resale Portal launched in January 2018 to make resale-flat transactions easier for buyers and sellers to complete themselves. Using data from 2017 onwards, quantify the business impact on agents and present it as a public-facing data story.

This notebook builds a data story from two official data.gov.sg CSV datasets:

- CEA Salespersons' Property Transaction Records (residential)
- HDB Resale Flat Prices from January 2017 onwards

The key measurement issue is that the two datasets count different units. HDB resale prices count **registered HDB resale transactions**. CEA records count **CEA salesperson-side records**. One resale flat can therefore produce zero, one, or two CEA rows depending on whether the buyer side, seller side, both sides, or neither side used an agent.

For that reason, the main metric used here is an **agent-side record rate**: CEA buyer/seller CEA salesperson-side records divided by the maximum possible buyer/seller sides in the HDB resale market. This is a practical opportunity proxy rather than an exact count of unique flats with agents.


## Analytical Flow

The story is framed as a before-and-after descriptive analysis. The HDB Resale Portal is treated as an important market event from January 2018, while the 2017 period provides a pre-portal baseline.

The notebook follows this sequence:

1. Load the official CEA and HDB resale datasets.
2. Preprocess dates and filter to HDB resale buyer/seller CEA salesperson-side records.
3. Build monthly and yearly summaries to understand the market over time.
4. Compare observed CEA salesperson-side records with a 2017 baseline rate.
5. Check data-quality issues and outliers before interpreting the figures.
6. Break the results down by buyer/seller side, town, flat type, and price band.
7. Separate transaction opportunity from revenue exposure using simple scenarios.
8. End with a public-facing conclusion and caveats.

The analysis avoids claiming that every post-2018 movement was caused by the portal. Other factors such as resale-market cycles, policy changes, COVID-era timing effects, interest rates, and changing flat mix also shaped the period after 2018.


In [ ]:
# If this notebook is run in a fresh environment, install the common analysis stack first.
# You can uncomment the next line if imports fail.
# %pip install pandas requests matplotlib seaborn tabulate

from pathlib import Path
import math
import textwrap

import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd().resolve()
SECTION_DIR = ROOT.parent if ROOT.name == "codes" else ROOT
CODES_DIR = ROOT if ROOT.name == "codes" else ROOT / "codes"

DATA_DIR = CODES_DIR / "data"
OUTPUT_DIR = CODES_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "cea": {
        "dataset_id": "d_ee7e46d3c57f7865790704632b0aef71",
        "filename": "cea_salespersons_property_transaction_records_residential.csv",
        "label": "CEA salesperson residential transaction records",
    },
    "hdb": {
        "dataset_id": "d_8b84c4ee58e3cfc0ece0d773c8ca6abc",
        "filename": "resale_flat_prices_2017_onwards.csv",
        "label": "HDB resale flat prices from Jan 2017 onwards",
    },
}

API_BASE = "https://api-open.data.gov.sg/v1/public/api/datasets"

In [ ]:
def get_download_url(dataset_id: str) -> str:
    """Ask data.gov.sg for a fresh temporary CSV download URL."""
    endpoint = f"{API_BASE}/{dataset_id}/poll-download"
    response = requests.get(endpoint, timeout=60)
    response.raise_for_status()
    payload = response.json()
    status = payload.get("data", {}).get("status")
    if status != "DOWNLOAD_SUCCESS":
        raise RuntimeError(f"Download was not ready for {dataset_id}: {payload}")
    return payload["data"]["url"]


def download_csv(dataset_key: str, force: bool = False) -> Path:
    """Download one official CSV into the local data folder."""
    spec = DATASETS[dataset_key]
    destination = DATA_DIR / spec["filename"]
    if destination.exists() and not force:
        print(f"Using existing file: {destination}")
        return destination

    print(f"Downloading {spec['label']}...")
    download_url = get_download_url(spec["dataset_id"])
    with requests.get(download_url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with destination.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    print(f"Saved: {destination} ({destination.stat().st_size / 1_000_000:,.1f} MB)")
    return destination


cea_path = download_csv("cea", force=False)
hdb_path = download_csv("hdb", force=False)

In [ ]:
cea = pd.read_csv(cea_path)
hdb = pd.read_csv(hdb_path)

print("CEA shape:", cea.shape)
print("HDB shape:", hdb.shape)
display(cea.head())
display(hdb.head())

## Preprocessing and First Look at the Data

Before calculating impact, the raw files are checked at a simple structural level: available columns, date coverage, missing values in important fields, and the volume of records by year and month.

This step makes the notebook more realistic because it shows whether the data is suitable for time-series analysis before moving into conclusions.


In [ ]:
# Preprocessing checks: structure, missingness, date coverage, and monthly/yearly record volumes.
raw_data_checks = {
    "cea_columns": cea.columns.tolist(),
    "hdb_columns": hdb.columns.tolist(),
    "cea_rows": len(cea),
    "hdb_rows": len(hdb),
    "cea_duplicate_rows": int(cea.duplicated().sum()),
    "hdb_duplicate_rows": int(hdb.duplicated().sum()),
}

cea_date_preview = pd.to_datetime(cea["transaction_date"], format="%b-%Y", errors="coerce")
hdb_date_preview = pd.to_datetime(hdb["month"], errors="coerce")

coverage_summary = pd.DataFrame(
    [
        {
            "dataset": "CEA salesperson records",
            "first_month": cea_date_preview.min(),
            "latest_month": cea_date_preview.max(),
            "rows": len(cea),
            "unparsed_months": int(cea_date_preview.isna().sum()),
        },
        {
            "dataset": "HDB resale flat prices",
            "first_month": hdb_date_preview.min(),
            "latest_month": hdb_date_preview.max(),
            "rows": len(hdb),
            "unparsed_months": int(hdb_date_preview.isna().sum()),
        },
    ]
)

key_missingness = pd.DataFrame(
    {
        "CEA missing values": cea[["transaction_date", "property_type", "transaction_type", "represented", "town"]].isna().sum(),
        "HDB missing values": hdb[["month", "town", "flat_type", "resale_price"]].isna().sum(),
    }
).fillna(0).astype(int)

cea_raw_monthly = cea_date_preview.dt.to_period("M").value_counts().sort_index().rename("cea_rows")
hdb_raw_monthly = hdb_date_preview.dt.to_period("M").value_counts().sort_index().rename("hdb_rows")
raw_monthly_profile = pd.concat([cea_raw_monthly, hdb_raw_monthly], axis=1).fillna(0).astype(int)
raw_yearly_profile = raw_monthly_profile.groupby(raw_monthly_profile.index.year).sum()

print("Raw data checks:")
display(raw_data_checks)
print("Dataset coverage:")
display(coverage_summary)
print("Missing values in key fields:")
display(key_missingness)
print("Year-by-year raw record profile:")
display(raw_yearly_profile)
print("Month-by-month raw record profile, latest 18 months:")
display(raw_monthly_profile.tail(18))


In [ ]:
# Parse dates and focus the CEA dataset on CEA-recorded HDB resale buyer/seller records.
cea["month"] = pd.to_datetime(cea["transaction_date"], format="%b-%Y")
hdb["month"] = pd.to_datetime(hdb["month"])

cea_hdb_resale = cea.loc[
    (cea["property_type"] == "HDB")
    & (cea["transaction_type"] == "RESALE")
    & (cea["represented"].isin(["BUYER", "SELLER"]))
].copy()

print("CEA HDB resale agent-side rows:", len(cea_hdb_resale))
display(cea_hdb_resale[["transaction_date", "represented", "town", "salesperson_reg_num"]].head())

In [ ]:
agent_monthly = (
    cea_hdb_resale
    .groupby(["month", "represented"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"BUYER": "buyer_agent_sides", "SELLER": "seller_agent_sides"})
)
for col in ["buyer_agent_sides", "seller_agent_sides"]:
    if col not in agent_monthly:
        agent_monthly[col] = 0
agent_monthly["agent_sides"] = agent_monthly["buyer_agent_sides"] + agent_monthly["seller_agent_sides"]

hdb_monthly = hdb.groupby("month").size().rename("hdb_resale_transactions").to_frame()

monthly = hdb_monthly.join(agent_monthly, how="left").fillna(0)
monthly["max_possible_sides"] = monthly["hdb_resale_transactions"] * 2
monthly["agent_side_record_rate"] = monthly["agent_sides"] / monthly["max_possible_sides"]
monthly["buyer_side_record_rate"] = monthly["buyer_agent_sides"] / monthly["hdb_resale_transactions"]
monthly["seller_side_record_rate"] = monthly["seller_agent_sides"] / monthly["hdb_resale_transactions"]

# Backward-compatible aliases for earlier chart code. Use the *_record_rate
# columns in explanation because CEA rows are records, not unique sides.
monthly["agent_side_penetration"] = monthly["agent_side_record_rate"]
monthly["buyer_side_penetration"] = monthly["buyer_side_record_rate"]
monthly["seller_side_penetration"] = monthly["seller_side_record_rate"]
monthly["year"] = monthly.index.year

display(monthly.tail())

In [ ]:
baseline_2017 = (
    monthly.loc[monthly["year"] == 2017, "agent_sides"].sum()
    / monthly.loc[monthly["year"] == 2017, "max_possible_sides"].sum()
)

annual = monthly.groupby("year", as_index=True).agg(
    hdb_resale_transactions=("hdb_resale_transactions", "sum"),
    buyer_agent_sides=("buyer_agent_sides", "sum"),
    seller_agent_sides=("seller_agent_sides", "sum"),
    agent_sides=("agent_sides", "sum"),
    max_possible_sides=("max_possible_sides", "sum"),
)
annual["agent_side_record_rate"] = annual["agent_sides"] / annual["max_possible_sides"]
annual["buyer_side_record_rate"] = annual["buyer_agent_sides"] / annual["hdb_resale_transactions"]
annual["seller_side_record_rate"] = annual["seller_agent_sides"] / annual["hdb_resale_transactions"]
annual["agent_side_penetration"] = annual["agent_side_record_rate"]  # compatibility alias
annual["expected_sides_at_2017_rate"] = annual["max_possible_sides"] * baseline_2017
annual["side_gap_vs_2017_rate"] = annual["agent_sides"] - annual["expected_sides_at_2017_rate"]
annual["record_rate_change_vs_2017_pp"] = (annual["agent_side_record_rate"] - baseline_2017) * 100
annual["penetration_change_vs_2017_pp"] = annual["record_rate_change_vs_2017_pp"]  # compatibility alias

annual_display = annual.copy()
annual_display["agent_side_record_rate"] = annual_display["agent_side_record_rate"].map(lambda x: f"{x:.1%}")
annual_display["record_rate_change_vs_2017_pp"] = annual_display["record_rate_change_vs_2017_pp"].map(lambda x: f"{x:+.1f} pp")
annual_display[[
    "hdb_resale_transactions",
    "agent_sides",
    "agent_side_record_rate",
    "side_gap_vs_2017_rate",
    "record_rate_change_vs_2017_pp",
]]

# Use only complete years for annual management conclusions. The latest available
# calendar year may be partial and should be handled as monitoring, not a full-year finding.
months_per_year = monthly.groupby("year").size()
complete_years = months_per_year[months_per_year == 12].index.tolist()
latest_complete_year = max(complete_years)
print(f"Latest complete year for annual comparison: {latest_complete_year}")
print(f"Latest month in dataset: {monthly.index.max().strftime('%Y-%m')}")


## Year-by-Year and Month-by-Month Analysis Dataset

After filtering and aligning the datasets, the analysis dataset is reviewed at two levels:

- **Year by year:** suitable for headline conclusions because it smooths monthly timing noise.
- **Month by month:** useful for spotting seasonality, partial-year effects, and unusual periods.

The 2017 agent-side record rate is used as the baseline because it is the full calendar year before the portal launch.


In [ ]:
year_by_year_summary = annual.copy()
year_by_year_summary["agent_side_record_rate_pct"] = year_by_year_summary["agent_side_record_rate"] * 100
year_by_year_summary["buyer_side_record_rate_pct"] = year_by_year_summary["buyer_side_record_rate"] * 100
year_by_year_summary["seller_side_record_rate_pct"] = year_by_year_summary["seller_side_record_rate"] * 100

month_by_month_summary = monthly.reset_index().rename(columns={"month": "calendar_month"})
month_by_month_summary["calendar_month"] = month_by_month_summary["calendar_month"].dt.to_period("M").astype(str)
month_by_month_summary["agent_side_record_rate_pct"] = month_by_month_summary["agent_side_record_rate"] * 100
month_by_month_summary["buyer_side_record_rate_pct"] = month_by_month_summary["buyer_side_record_rate"] * 100
month_by_month_summary["seller_side_record_rate_pct"] = month_by_month_summary["seller_side_record_rate"] * 100

print(f"2017 baseline agent-side record rate: {baseline_2017:.1%}")
print("Year-by-year analysis summary:")
display(
    year_by_year_summary[[
        "hdb_resale_transactions",
        "agent_sides",
        "agent_side_record_rate_pct",
        "buyer_side_record_rate_pct",
        "seller_side_record_rate_pct",
        "side_gap_vs_2017_rate",
        "record_rate_change_vs_2017_pp",
    ]]
)
print("Month-by-month analysis summary, latest 18 months:")
display(
    month_by_month_summary[[
        "calendar_month",
        "hdb_resale_transactions",
        "agent_sides",
        "agent_side_record_rate_pct",
        "buyer_side_record_rate_pct",
        "seller_side_record_rate_pct",
    ]].tail(18)
)


## Data-Quality Checks for the Agent-Side Proxy

Before interpreting the agent-side record rate, check for periods where CEA records exceed the theoretical number of buyer/seller sides. These cases do not make the analysis unusable, but they show why the metric should be presented as a **record-rate proxy** rather than a literal penetration share.

The practical reading is that annual aggregation is more stable for the public story, while monthly charts should be used to diagnose timing or recording distortions.


In [ ]:
quality_checks = monthly.assign(
    agent_side_record_rate_pct=monthly["agent_side_record_rate"] * 100,
    buyer_side_record_rate_pct=monthly["buyer_side_record_rate"] * 100,
    seller_side_record_rate_pct=monthly["seller_side_record_rate"] * 100,
)

anomalous_months = quality_checks.loc[
    (quality_checks["agent_side_record_rate"] > 1)
    | (quality_checks["buyer_side_record_rate"] > 1)
    | (quality_checks["seller_side_record_rate"] > 1),
    [
        "hdb_resale_transactions",
        "buyer_agent_sides",
        "seller_agent_sides",
        "agent_sides",
        "agent_side_record_rate_pct",
        "buyer_side_record_rate_pct",
        "seller_side_record_rate_pct",
    ],
].sort_values("agent_side_record_rate_pct", ascending=False)

print("Months where record-rate proxy exceeds a literal 100% interpretation:", len(anomalous_months))
display(anomalous_months.head(12))


## Outlier Review and Side-Level Breakdown

The outlier review separates two ideas:

- months where the record-rate proxy is unusually high or low compared with the rest of the series;
- whether the buyer side or seller side contributes more to those unusual movements.

This is useful before writing the final narrative because it prevents one unusual month from being overinterpreted as a long-term market shift.


In [ ]:
outlier_base = monthly.copy()
rate_q1 = outlier_base["agent_side_record_rate"].quantile(0.25)
rate_q3 = outlier_base["agent_side_record_rate"].quantile(0.75)
rate_iqr = rate_q3 - rate_q1
lower_fence = rate_q1 - 1.5 * rate_iqr
upper_fence = rate_q3 + 1.5 * rate_iqr

rate_outliers = outlier_base.loc[
    (outlier_base["agent_side_record_rate"] < lower_fence)
    | (outlier_base["agent_side_record_rate"] > upper_fence)
].copy()
rate_outliers["agent_side_record_rate_pct"] = rate_outliers["agent_side_record_rate"] * 100
rate_outliers["buyer_share_of_agent_sides_pct"] = (
    rate_outliers["buyer_agent_sides"] / rate_outliers["agent_sides"].replace(0, pd.NA) * 100
)
rate_outliers["seller_share_of_agent_sides_pct"] = (
    rate_outliers["seller_agent_sides"] / rate_outliers["agent_sides"].replace(0, pd.NA) * 100
)

side_breakdown_by_year = annual.copy()
side_breakdown_by_year["buyer_share_of_agent_sides_pct"] = side_breakdown_by_year["buyer_agent_sides"] / side_breakdown_by_year["agent_sides"] * 100
side_breakdown_by_year["seller_share_of_agent_sides_pct"] = side_breakdown_by_year["seller_agent_sides"] / side_breakdown_by_year["agent_sides"] * 100
side_breakdown_by_year["buyer_minus_seller_record_rate_pp"] = (
    side_breakdown_by_year["buyer_side_record_rate"] - side_breakdown_by_year["seller_side_record_rate"]
) * 100

print("IQR fences for monthly agent-side record rate:")
print(f"Lower fence: {lower_fence:.1%}; upper fence: {upper_fence:.1%}")
print("Outlier months by agent-side record rate:")
display(
    rate_outliers[[
        "hdb_resale_transactions",
        "agent_sides",
        "agent_side_record_rate_pct",
        "buyer_share_of_agent_sides_pct",
        "seller_share_of_agent_sides_pct",
    ]].sort_values("agent_side_record_rate_pct", ascending=False)
)
print("Buyer/seller breakdown by year:")
display(
    side_breakdown_by_year[[
        "buyer_agent_sides",
        "seller_agent_sides",
        "buyer_share_of_agent_sides_pct",
        "seller_share_of_agent_sides_pct",
        "buyer_minus_seller_record_rate_pp",
    ]]
)


## Event Framing and Interpretation

The portal launch is a useful event marker, but the analysis should not attribute every post-2018 movement to the portal. Resale demand, policy settings, COVID-era timing distortions, and market cycles all changed after 2018.

The interpretation used here is a **before/after descriptive analysis**. A causal version would need controls, comparison groups, or an explicit interrupted-time-series or difference-in-differences design.


In [ ]:
causal_caveat = {
    "event": "HDB Resale Portal launch",
    "event_month": "2018-01",
    "analysis_type": "descriptive before/after with 2017 baseline",
    "not_claimed": "causal effect of the portal",
    "potential_confounders": [
        "resale market cycle",
        "policy changes",
        "COVID-era completion and recording timing",
        "interest-rate and affordability conditions",
        "changes in flat mix and estate mix",
    ],
}
causal_caveat


## Opportunity Impact Versus Revenue Exposure

CEA salesperson-side records measure transaction opportunity, not commission revenue. To discuss revenue, this notebook uses scenarios rather than a single point estimate.

The table below applies illustrative commission assumptions to the baseline gap. It is a sensitivity check, not a claim of exact agent income gained or lost.


In [ ]:
# Scenario-based revenue exposure. These assumptions are illustrative and should
# be replaced with validated commission assumptions before external use.
annual_price = hdb.assign(year=hdb["month"].dt.year).groupby("year")["resale_price"].median().rename("median_resale_price")
annual_revenue = annual.join(annual_price)

commission_scenarios = {
    "low_0_5pct": 0.005,
    "base_1_0pct": 0.010,
    "high_2_0pct": 0.020,
}
for name, rate in commission_scenarios.items():
    annual_revenue[f"revenue_gap_{name}"] = annual_revenue["side_gap_vs_2017_rate"] * annual_revenue["median_resale_price"] * rate

scenario_cols = ["hdb_resale_transactions", "agent_sides", "side_gap_vs_2017_rate", "median_resale_price"] + [f"revenue_gap_{name}" for name in commission_scenarios]
display(annual_revenue.loc[complete_years, scenario_cols])


## Figure 1: Monthly Agent-Side Record Rate

This figure shows the month-by-month record-rate proxy with January 2018 marked as the portal launch. It is best read as a timing chart, not as the only basis for the conclusion.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(monthly.index, monthly["agent_side_penetration"] * 100, linewidth=2.5, color="#0F766E")
ax.axvline(pd.Timestamp("2018-01-01"), color="#B91C1C", linestyle="--", linewidth=1.5)
ax.text(pd.Timestamp("2018-02-01"), ax.get_ylim()[1] * 0.95, "HDB Resale Portal\nlaunches", color="#B91C1C", va="top")
ax.set_title("Share of HDB resale buyer/seller sides represented by agents")
ax.set_ylabel("agent-side record rate (%)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(lambda value, pos: f"{value:.0f}%")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "monthly_agent_side_penetration.png", dpi=200, bbox_inches="tight")
plt.show()

## Figure 2: Agent-Side Records Versus the 2017 Baseline

This figure translates the rate change into a count of CEA salesperson-side records above or below what would have been expected if the 2017 rate had continued. It is the clearest business-impact chart because it connects the public metric to transaction opportunity.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_df = annual.reset_index()
sns.barplot(data=plot_df, x="year", y="side_gap_vs_2017_rate", color="#2563EB", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Agent-side volume compared with the 2017 pre-portal rate")
ax.set_xlabel("")
ax.set_ylabel("Agent sides above / below 2017-rate expectation")
ax.yaxis.set_major_formatter(lambda value, pos: f"{value:,.0f}")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "annual_agent_side_gap_vs_2017.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Town-level view: where did agent-side record rate move most from 2017 to the latest complete year?
months_per_year = monthly.groupby("year").size()
complete_years = months_per_year[months_per_year == 12].index.tolist()
latest_complete_year = max(complete_years)

agent_town_year = (
    cea_hdb_resale.groupby([cea_hdb_resale["month"].dt.year.rename("year"), "town"])
    .size()
    .rename("agent_sides")
    .reset_index()
)
hdb_town_year = (
    hdb.groupby([hdb["month"].dt.year.rename("year"), "town"])
    .size()
    .rename("hdb_resale_transactions")
    .reset_index()
)
town_year = hdb_town_year.merge(agent_town_year, on=["year", "town"], how="left").fillna({"agent_sides": 0})
town_year["agent_side_penetration"] = town_year["agent_sides"] / (2 * town_year["hdb_resale_transactions"])

town_compare = town_year.loc[town_year["year"].isin([2017, latest_complete_year])].pivot(
    index="town", columns="year", values="agent_side_penetration"
)
town_compare["change_pp"] = (town_compare[latest_complete_year] - town_compare[2017]) * 100
town_compare = town_compare.dropna().sort_values("change_pp")
display(town_compare.head(10))
display(town_compare.tail(10))

## Breakdowns for Interpreting the Change

The overall trend is only the starting point. The next step is to identify where the change appears more concentrated.

Useful cuts include:

- **Buyer vs seller side:** whether one side shows more movement in CEA salesperson-side records.
- **Town:** where changes differ geographically.
- **Flat type:** whether common flat types dominate the resale market context.
- **Price band:** whether higher-value resale transactions look different from lower-value transactions.

CEA and HDB records cannot be matched deal by deal, so flat-type and price-band tables are market-context breakdowns rather than exact agent-rate segments.


In [ ]:
# Flat-type and price-band segmentation for the latest complete year.
hdb_seg = hdb.copy()
hdb_seg["year"] = hdb_seg["month"].dt.year
hdb_seg["price_band"] = pd.qcut(
    hdb_seg["resale_price"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop",
)

hdb_flat_year = (
    hdb_seg.groupby(["year", "flat_type"])
    .size()
    .rename("hdb_resale_transactions")
    .reset_index()
)
cea_flat_proxy = cea_hdb_resale.copy()
cea_flat_proxy["year"] = cea_flat_proxy["month"].dt.year

# CEA does not include flat_type or resale price, so flat-type / price-band cuts
# are market-context segments rather than direct CEA-matched agent-rate segments.
flat_mix_latest = hdb_flat_year.loc[hdb_flat_year["year"] == latest_complete_year].sort_values("hdb_resale_transactions", ascending=False)
price_band_latest = (
    hdb_seg.loc[hdb_seg["year"] == latest_complete_year]
    .groupby("price_band", observed=False)
    .agg(
        hdb_resale_transactions=("resale_price", "size"),
        median_resale_price=("resale_price", "median"),
    )
    .reset_index()
)

display(flat_mix_latest)
display(price_band_latest)


## Complete Years for Conclusions, Partial Years for Monitoring

Because the data can include partial years, annual conclusions should use complete calendar years only. Partial-year data can still be useful, but it should be labelled as monitoring data and kept out of the headline conclusion.


In [ ]:
annual_conclusion = annual.loc[complete_years].copy()
partial_years = sorted(set(monthly["year"]) - set(complete_years))

print("Complete years used for headline annual conclusions:", complete_years)
print("Partial years reserved for monitoring only:", partial_years)
display(annual_conclusion[["hdb_resale_transactions", "agent_sides", "agent_side_record_rate", "side_gap_vs_2017_rate"]].tail())


## Conclusion and Public Data Story

**Working headline:** The HDB Resale Portal appears to have changed the shape of agent opportunity rather than removing agents from the resale process.

**Main read:** compare each complete year with the 2017 pre-portal baseline. If the agent-side record rate falls below the baseline, the gap estimates the number of buyer/seller CEA salesperson-side records that did not appear relative to the pre-portal pattern. If it rises above the baseline, the market generated more CEA salesperson-side records than the 2017-rate expectation.

**How to read the figures:** use annual charts for the main public message and monthly charts for detail. The monthly chart is useful because it shows timing around January 2018, but the annual table is more reliable for conclusions.

**Breakdown read:** buyer/seller side and town-level views show where the change is uneven. Flat-type and price-band summaries add market context, but they should not be described as exact agent-use breakdowns because the CEA and HDB datasets do not share a transaction ID.

**Revenue read:** transaction opportunity and commission revenue are related but not identical. Any revenue estimate should be presented as a scenario because actual commission terms are not observed in the public data.

**Caveats to state plainly:** CEA records count salesperson-side records, not unique deals or confirmed commissions. The portal launch is an event marker in this notebook; the analysis is descriptive and does not prove causality by itself.

**Public-facing closing:** the evidence supports a measured story: self-service became easier after January 2018, but the public data still shows substantial CEA salesperson-side activity in registered HDB resale transactions. The business impact is best described through the change in agent-side record rate against the 2017 baseline, supported by breakdowns that show where the pattern differs across sides and towns.
